<a href="https://colab.research.google.com/github/kavitha-kaliyaperumal/Deep-Fake-Detection/blob/main/deepfake_brain_mri_vit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Deepfake Brain MRI Detection — Vision Transformer (ViT)

**Architecture:** Vision Transformer (ViT) — Google 2020 `"An Image is Worth 16×16 Words"`

### 🧩 How ViT works (in plain English)
```
Input MRI (224×224)
       ↓
Split into 14×14 grid of 16×16 patches  →  196 patches
       ↓
Each patch → linear embedding (like a token in BERT)
       ↓
Add [CLS] token + positional encodings
       ↓
Pass through N × Transformer Encoder blocks
  (Multi-Head Self-Attention + Feed Forward + LayerNorm)
       ↓
[CLS] token output → MLP head → Real / Fake
```

### 📋 Notebook Outline
1. Install & imports
2. Synthetic MRI dataset
3. ViT from scratch (understand every layer)
4. Pretrained ViT fine-tuning (`vit_b_16`)
5. Attention map visualization
6. Training & evaluation
7. Compare ViT vs CNN

> ⚠️ **Runtime:** `Runtime > Change runtime type > T4 GPU`

## 📦 STEP 1 — Install & Imports

In [ ]:
!pip install -q timm einops

import os, math, random
from pathlib import Path
from PIL import Image

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

import timm
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 🗂️ STEP 2 — Generate Synthetic Brain MRI Dataset

In [ ]:
from scipy.ndimage import gaussian_filter

DATA_DIR = Path('mri_dataset')
REAL_DIR = DATA_DIR / 'real'
FAKE_DIR = DATA_DIR / 'fake'
REAL_DIR.mkdir(parents=True, exist_ok=True)
FAKE_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE  = 224
N_SAMPLES = 400   # per class — increase for better training

def make_mri(seed, is_fake=False):
    """Synthetic grayscale brain MRI-like image."""
    rng = np.random.RandomState(seed)

    # Layered frequency components mimic MRI texture
    img = sum(
        gaussian_filter(rng.randn(IMG_SIZE, IMG_SIZE), sigma=s) * w
        for s, w in [(12, 1.0), (4, 0.4), (1.5, 0.15)]
    )

    # Elliptical brain mask
    Y, X = np.ogrid[:IMG_SIZE, :IMG_SIZE]
    cx, cy = IMG_SIZE // 2, IMG_SIZE // 2
    mask = ((X - cx) / 95)**2 + ((Y - cy) / 105)**2 <= 1
    img *= mask

    if is_fake:
        # Simulate GAN artifacts
        # 1. High-freq checkerboard (deconv artifact)
        checker = (np.indices((IMG_SIZE, IMG_SIZE)).sum(0) % 4 == 0).astype(float) * 0.12
        # 2. Blotchy texture inconsistency
        blotch  = gaussian_filter(rng.rand(IMG_SIZE, IMG_SIZE) > 0.85, sigma=3).astype(float) * 0.2
        # 3. Slight spectral shift
        img = img * 0.9 + rng.randn(IMG_SIZE, IMG_SIZE) * 0.08
        img += (checker + blotch) * mask

    img = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255
    return Image.fromarray(img.astype(np.uint8)).convert('RGB')

print('Generating dataset...')
for i in range(N_SAMPLES):
    make_mri(i,         is_fake=False).save(REAL_DIR / f'real_{i:04d}.png')
    make_mri(i + 50000, is_fake=True ).save(FAKE_DIR / f'fake_{i:04d}.png')
print(f'✅ {N_SAMPLES} real + {N_SAMPLES} fake images saved')

In [ ]:
# ── Visualize real vs fake ────────────────────────────────────
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
fig.suptitle('Synthetic Brain MRI — Real vs Fake', fontsize=13, fontweight='bold')

for i, (r, f) in enumerate(zip(
    sorted(REAL_DIR.glob('*.png'))[:6],
    sorted(FAKE_DIR.glob('*.png'))[:6]
)):
    axes[0, i].imshow(Image.open(r), cmap='gray'); axes[0, i].set_title('REAL', color='green', fontsize=9); axes[0, i].axis('off')
    axes[1, i].imshow(Image.open(f), cmap='gray'); axes[1, i].set_title('FAKE', color='red',   fontsize=9); axes[1, i].axis('off')

plt.tight_layout(); plt.show()

## 🧩 STEP 3 — ViT Internals: Understand Each Component

We build ViT piece by piece so you understand every layer.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3a.  PATCH EMBEDDING
# Split the image into fixed-size patches, then project each
# patch to a D-dimensional embedding vector.
# ═══════════════════════════════════════════════════════════════
class PatchEmbedding(nn.Module):
    """
    Splits an image into non-overlapping patches and embeds them.

    Args:
        img_size  : input image height/width (square assumed)
        patch_size: size of each square patch (e.g. 16 → 16×16 pixels)
        in_chans  : number of input channels (3 for RGB)
        embed_dim : output embedding dimension D
    """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2   # 196 for 224/16
        self.patch_size  = patch_size
        # A single Conv2d with kernel=patch_size and stride=patch_size
        # is equivalent to: split → flatten → linear
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: (B, C, H, W) → (B, D, H/P, W/P) → (B, N, D)
        x = self.proj(x)               # (B, D, 14, 14)
        x = x.flatten(2).transpose(1, 2)  # (B, 196, D)
        return x


# ═══════════════════════════════════════════════════════════════
# 3b.  MULTI-HEAD SELF-ATTENTION (MHSA)
# Each patch attends to every other patch — captures global
# dependencies that CNNs miss with local receptive fields.
# ═══════════════════════════════════════════════════════════════
class MultiHeadSelfAttention(nn.Module):
    """
    Standard scaled dot-product multi-head attention.

    Attention(Q,K,V) = softmax( QK^T / sqrt(d_k) ) V
    """
    def __init__(self, embed_dim=768, num_heads=12, dropout=0.0):
        super().__init__()
        assert embed_dim % num_heads == 0, 'embed_dim must be divisible by num_heads'
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5   # 1/sqrt(d_k)

        self.qkv   = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.proj  = nn.Linear(embed_dim, embed_dim)
        self.drop  = nn.Dropout(dropout)
        self.attn_weights = None  # stored for visualization

    def forward(self, x):
        B, N, D = x.shape
        # Compute Q, K, V in one matrix multiply then split
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)   # (3, B, heads, N, head_dim)
        q, k, v = qkv.unbind(0)             # each: (B, heads, N, head_dim)

        # Scaled dot-product attention
        attn = (q @ k.transpose(-2, -1)) * self.scale   # (B, heads, N, N)
        attn = attn.softmax(dim=-1)
        self.attn_weights = attn.detach()   # save for visualization
        attn = self.drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, D)
        return self.proj(x)


# ═══════════════════════════════════════════════════════════════
# 3c.  TRANSFORMER ENCODER BLOCK
# Pre-LayerNorm variant (more stable training than original paper)
# ═══════════════════════════════════════════════════════════════
class TransformerBlock(nn.Module):
    """
    One Transformer encoder layer:
      x = x + Attention(LayerNorm(x))
      x = x + MLP(LayerNorm(x))
    """
    def __init__(self, embed_dim=768, num_heads=12, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden),
            nn.GELU(),          # GELU outperforms ReLU in transformers
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))   # residual connection
        x = x + self.mlp(self.norm2(x))    # residual connection
        return x


# ═══════════════════════════════════════════════════════════════
# 3d.  FULL VISION TRANSFORMER
# ═══════════════════════════════════════════════════════════════
class VisionTransformer(nn.Module):
    """
    ViT for binary classification (real / fake MRI).

    Key hyperparameters:
      img_size  : 224
      patch_size: 16  → 196 patches
      embed_dim : 256 (small), 384 (medium), 768 (ViT-B)
      depth     : number of transformer blocks
      num_heads : attention heads (embed_dim must be divisible)
    """
    def __init__(
        self,
        img_size   = 224,
        patch_size = 16,
        in_chans   = 3,
        num_classes= 2,
        embed_dim  = 256,   # small model for Colab
        depth      = 6,
        num_heads  = 8,
        mlp_ratio  = 4.0,
        dropout    = 0.1,
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches

        # Learnable [CLS] token prepended to patch sequence
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        # Learnable positional encoding for each position (patches + CLS)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop  = nn.Dropout(dropout)

        # Stack of transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        # Classification head: use only [CLS] token output
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(embed_dim // 2, num_classes)
        )

        # Weight initialization
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        B = x.shape[0]

        # 1. Patch embedding
        x = self.patch_embed(x)                         # (B, 196, D)

        # 2. Prepend [CLS] token
        cls = repeat(self.cls_token, '1 1 d -> b 1 d', b=B)
        x   = torch.cat([cls, x], dim=1)                # (B, 197, D)

        # 3. Add positional encoding
        x = self.pos_drop(x + self.pos_embed)

        # 4. Transformer blocks
        for block in self.blocks:
            x = block(x)

        # 5. Layer norm
        x = self.norm(x)

        # 6. Classify from [CLS] token (index 0)
        return self.head(x[:, 0])

    def get_attention_map(self, x, block_idx=-1):
        """Extract attention weights from a specific block for visualization."""
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = repeat(self.cls_token, '1 1 d -> b 1 d', b=B)
        x = torch.cat([cls, x], dim=1)
        x = self.pos_drop(x + self.pos_embed)
        for i, block in enumerate(self.blocks):
            x = block(x)
            if i == (len(self.blocks) + block_idx) % len(self.blocks):
                return block.attn.attn_weights   # (B, heads, N+1, N+1)
        return None


# ── Instantiate and inspect ──────────────────────────────────
vit_scratch = VisionTransformer(
    img_size=224, patch_size=16, embed_dim=256,
    depth=6, num_heads=8, dropout=0.1
).to(DEVICE)

total = sum(p.numel() for p in vit_scratch.parameters())
print(f'✅ ViT (from scratch) | Parameters: {total:,}')

# Quick shape test
dummy = torch.randn(2, 3, 224, 224).to(DEVICE)
out   = vit_scratch(dummy)
print(f'   Input shape : {list(dummy.shape)}')
print(f'   Output shape: {list(out.shape)}  (B=2, classes=2)')

## 🎨 STEP 4 — Visualize Patch Tokenization

In [ ]:
sample_img = Image.open(sorted(REAL_DIR.glob('*.png'))[0]).convert('RGB')
img_np = np.array(sample_img)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Vision Transformer: Patch Tokenization', fontsize=13, fontweight='bold')

# 1. Original image
axes[0].imshow(img_np, cmap='gray')
axes[0].set_title('① Original MRI (224×224)', fontsize=10)
axes[0].axis('off')

# 2. Patch grid overlay
axes[1].imshow(img_np, cmap='gray')
P = 16
for y in range(0, 224, P):
    for x in range(0, 224, P):
        rect = patches.Rectangle((x, y), P, P, linewidth=0.5,
                                   edgecolor='cyan', facecolor='none')
        axes[1].add_patch(rect)
axes[1].set_title(f'② Split into {(224//P)**2} patches of {P}×{P}px', fontsize=10)
axes[1].axis('off')

# 3. Display first 16 patches as tokens
grid = np.zeros((P*2, P*8, 3), dtype=np.uint8)
for i in range(16):
    r, c = (i // 8), (i % 8)
    py, px = (i // (224 // P)) * P, (i % (224 // P)) * P
    patch = img_np[py:py+P, px:px+P]
    if patch.shape[:2] == (P, P):
        grid[r*P:(r+1)*P, c*P:(c+1)*P] = patch
axes[2].imshow(grid)
axes[2].set_title('③ First 16 patch "tokens"', fontsize=10)
axes[2].axis('off')

plt.tight_layout(); plt.show()
print('Each patch is flattened (16×16×3 = 768 values) and projected to embed_dim=256.')

## 🔧 STEP 5 — Dataset & DataLoaders

In [ ]:
class MRIDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None):
        self.transform = transform
        self.samples = []
        for p in sorted(Path(real_dir).glob('*.png')): self.samples.append((str(p), 0))
        for p in sorted(Path(fake_dir).glob('*.png')): self.samples.append((str(p), 1))
        random.shuffle(self.samples)

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

# ViT benefits from stronger augmentation
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

full_ds = MRIDataset(REAL_DIR, FAKE_DIR, transform=train_tf)
n = len(full_ds)
n_train, n_val = int(0.7*n), int(0.15*n)
train_ds, val_ds, test_ds = random_split(full_ds, [n_train, n_val, n - n_train - n_val])

val_ds.dataset.transform  = val_tf
test_ds.dataset.transform = val_tf

BATCH = 32
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

## 🏋️ STEP 6 — Train ViT from Scratch

In [ ]:
def train_model(model, train_loader, val_loader, epochs=20, lr=3e-4, name='model'):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)   # label smoothing helps ViT
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    # Warmup + cosine decay — standard ViT schedule
    warmup = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=5)
    cosine = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - 5)
    scheduler = optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], milestones=[5])

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_acc = 0

    for epoch in range(epochs):
        # Train
        model.train()
        tl, tc, tt = 0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
            optimizer.step()
            with torch.no_grad():
                out = model(imgs)
                tl += loss.item(); tc += (out.argmax(1)==labels).sum().item(); tt += labels.size(0)
        scheduler.step()

        # Validate
        model.eval()
        vl, vc, vt = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = model(imgs)
                vl += criterion(out, labels).item(); vc += (out.argmax(1)==labels).sum().item(); vt += labels.size(0)

        ta, va = tc/tt, vc/vt
        history['train_loss'].append(tl/len(train_loader)); history['val_loss'].append(vl/len(val_loader))
        history['train_acc'].append(ta);                    history['val_acc'].append(va)

        if va > best_acc:
            best_acc = va
            torch.save(model.state_dict(), f'best_{name}.pth')

        lr_now = scheduler.get_last_lr()[0]
        print(f'[{epoch+1:2d}/{epochs}] LR={lr_now:.5f} | '
              f'Train {ta:.3f} | Val {va:.3f} {"⭐" if va==best_acc else ""}')

    print(f'\n✅ Best Val Acc: {best_acc:.4f}')
    return history

print('🏋️ Training ViT from scratch (6 layers, embed_dim=256)...')
print('   Expected time: ~3–5 min on T4 GPU\n')
scratch_history = train_model(vit_scratch, train_loader, val_loader, epochs=20, lr=3e-4, name='vit_scratch')

## 🤖 STEP 7 — Pretrained ViT Fine-Tuning (via `timm`)

A pretrained **ViT-Small** (`vit_small_patch16_224`) fine-tuned on your MRI data will almost always outperform training from scratch — especially with limited data.

In [ ]:
import timm

# Load pretrained ViT-Small (22M params, ImageNet-21k)
vit_pretrained = timm.create_model(
    'vit_small_patch16_224',
    pretrained=True,
    num_classes=2        # replace head for binary classification
).to(DEVICE)

# ── Fine-tuning strategy: 3 phases ──────────────────────────
# Phase 1: Freeze backbone, train head only (fast)
# Phase 2: Unfreeze all, low LR (full fine-tune)

def set_requires_grad(model, backbone_requires=False):
    for name, param in model.named_parameters():
        if 'head' in name:
            param.requires_grad = True
        else:
            param.requires_grad = backbone_requires

# Phase 1: head only
set_requires_grad(vit_pretrained, backbone_requires=False)
trainable = sum(p.numel() for p in vit_pretrained.parameters() if p.requires_grad)
print(f'Phase 1 — Trainable params (head only): {trainable:,}')
print('Training head (5 epochs)...')
hist_p1 = train_model(vit_pretrained, train_loader, val_loader, epochs=5, lr=1e-3, name='vit_pretrained')

# Phase 2: full fine-tune
set_requires_grad(vit_pretrained, backbone_requires=True)
trainable = sum(p.numel() for p in vit_pretrained.parameters() if p.requires_grad)
print(f'\nPhase 2 — Trainable params (full): {trainable:,}')
print('Full fine-tuning (10 epochs, low LR)...')
hist_p2 = train_model(vit_pretrained, train_loader, val_loader, epochs=10, lr=1e-4, name='vit_pretrained')

## 📈 STEP 8 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ViT from Scratch — Training History', fontsize=13, fontweight='bold')

for ax, key, title in zip(axes, ['loss', 'acc'], ['Loss', 'Accuracy']):
    ax.plot(scratch_history[f'train_{key}'], label='Train', marker='o', ms=4)
    ax.plot(scratch_history[f'val_{key}'],   label='Val',   marker='s', ms=4)
    ax.set_title(f'{title} Curve'); ax.set_xlabel('Epoch'); ax.set_ylabel(title)
    ax.legend(); ax.grid(True, alpha=0.3)
    if key == 'acc': ax.set_ylim(0, 1)

plt.tight_layout(); plt.show()

## 🔬 STEP 9 — Evaluate on Test Set

In [ ]:
def evaluate(model, loader, model_path, title):
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()
    preds, labels, probs = [], [], []

    with torch.no_grad():
        for imgs, lbl in loader:
            out = model(imgs.to(DEVICE))
            p   = torch.softmax(out, 1)[:, 1].cpu().numpy()
            probs.extend(p)
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(lbl.numpy())

    auc = roc_auc_score(labels, probs)
    print(f'\n━━━ {title} ━━━')
    print(classification_report(labels, preds, target_names=['Real', 'Fake']))
    print(f'AUC-ROC: {auc:.4f}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(title, fontsize=12, fontweight='bold')

    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Real','Fake'], yticklabels=['Real','Fake'])
    axes[0].set_title('Confusion Matrix'); axes[0].set_ylabel('True'); axes[0].set_xlabel('Pred')

    fpr, tpr, _ = roc_curve(labels, probs)
    axes[1].plot(fpr, tpr, lw=2, label=f'AUC = {auc:.3f}', color='steelblue')
    axes[1].plot([0,1],[0,1],'k--'); axes[1].set_title('ROC Curve')
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()
    return preds, labels, probs

print('Evaluating ViT from scratch...')
evaluate(vit_scratch,    test_loader, 'best_vit_scratch.pth',    'ViT from Scratch')

print('Evaluating Pretrained ViT...')
evaluate(vit_pretrained, test_loader, 'best_vit_pretrained.pth', 'Pretrained ViT-Small (fine-tuned)')

## 👁️ STEP 10 — Attention Map Visualization

Unlike Grad-CAM, ViT gives us **raw attention weights** — we can directly see which patches the model focuses on.

In [ ]:
import matplotlib.cm as cm

def visualize_attention(model, img_tensor, title='Attention Map'):
    """
    Show per-head and mean attention from [CLS] to all patches.
    This reveals which brain regions the model attends to.
    """
    model.eval()
    with torch.no_grad():
        attn = model.get_attention_map(img_tensor.unsqueeze(0).to(DEVICE))
    # attn: (1, num_heads, N+1, N+1) — row 0 = [CLS] attending to all patches
    attn = attn[0, :, 0, 1:]   # (heads, 196) — CLS → patches
    P = int(math.sqrt(attn.shape[-1]))  # 14
    attn_maps = attn.reshape(-1, P, P).cpu().numpy()  # (heads, 14, 14)

    num_heads = attn_maps.shape[0]
    cols = num_heads + 2  # each head + original + mean
    fig, axes = plt.subplots(1, cols, figsize=(cols * 2.2, 3))
    fig.suptitle(title, fontsize=12, fontweight='bold')

    # Original image
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = (img_np * 0.5 + 0.5).clip(0, 1)
    axes[0].imshow(img_np); axes[0].set_title('Input MRI', fontsize=8); axes[0].axis('off')

    # Per-head attention
    for i, amap in enumerate(attn_maps):
        axes[i+1].imshow(amap, cmap='hot', interpolation='nearest')
        axes[i+1].set_title(f'Head {i+1}', fontsize=8); axes[i+1].axis('off')

    # Mean attention across heads
    mean_attn = attn_maps.mean(0)
    axes[-1].imshow(mean_attn, cmap='hot', interpolation='nearest')
    axes[-1].set_title('Mean Attention', fontsize=8, fontweight='bold'); axes[-1].axis('off')

    plt.tight_layout(); plt.show()

# Load and visualize for a real and a fake image
vit_scratch.load_state_dict(torch.load('best_vit_scratch.pth', map_location=DEVICE))

val_tf2 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

real_tensor = val_tf2(Image.open(sorted(REAL_DIR.glob('*.png'))[10]).convert('RGB'))
fake_tensor = val_tf2(Image.open(sorted(FAKE_DIR.glob('*.png'))[10]).convert('RGB'))

visualize_attention(vit_scratch, real_tensor, 'Attention Map — REAL MRI')
visualize_attention(vit_scratch, fake_tensor, 'Attention Map — FAKE MRI')
print('🔥 Brighter = more attention. Compare where the model focuses for real vs fake.')

## 📊 STEP 11 — Positional Embedding Similarity Visualization

Inspect what the ViT learned about spatial structure.

In [ ]:
# Cosine similarity between each patch position and every other
pos_embed = vit_scratch.pos_embed[0, 1:, :].detach().cpu()  # (196, D) — skip CLS
pos_embed_norm = F.normalize(pos_embed, dim=-1)
sim_matrix = (pos_embed_norm @ pos_embed_norm.T).numpy()    # (196, 196)

P = 14  # 224//16
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Learned Positional Embedding Similarity', fontsize=13, fontweight='bold')

# Full similarity matrix
axes[0].imshow(sim_matrix, cmap='RdYlBu_r', vmin=-1, vmax=1)
axes[0].set_title('196×196 similarity matrix', fontsize=9)
axes[0].set_xlabel('Patch index'); axes[0].set_ylabel('Patch index')

# Similarity to patch 98 (center)
center = P * P // 2
sim_center = sim_matrix[center].reshape(P, P)
im = axes[1].imshow(sim_center, cmap='RdYlBu_r', vmin=-1, vmax=1)
axes[1].set_title(f'Similarity to patch {center} (center)', fontsize=9)
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], shrink=0.8)

# Similarity to patch 0 (top-left)
sim_corner = sim_matrix[0].reshape(P, P)
im2 = axes[2].imshow(sim_corner, cmap='RdYlBu_r', vmin=-1, vmax=1)
axes[2].set_title('Similarity to patch 0 (top-left)', fontsize=9)
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.tight_layout(); plt.show()
print('Nearby patches have higher similarity → the model learned spatial structure!')

## 🔍 STEP 12 — Single Image Inference

In [ ]:
def predict_mri(image_path, model, model_path, device=DEVICE):
    """Predict real vs fake for a single MRI image."""
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    img  = Image.open(image_path).convert('RGB')
    tens = tf(img).unsqueeze(0).to(device)

    with torch.no_grad():
        out  = model(tens)
        prob = torch.softmax(out, 1)[0]
        pred = out.argmax(1).item()

    label = '🔴 FAKE' if pred == 1 else '🟢 REAL'
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(np.array(img), cmap='gray'); axes[0].set_title('Input MRI'); axes[0].axis('off')
    bars = axes[1].bar(['Real', 'Fake'], prob.cpu().numpy(), color=['green', 'red'], alpha=0.8)
    axes[1].set_ylim(0, 1); axes[1].set_ylabel('Probability')
    axes[1].set_title(f'Prediction: {label}', fontweight='bold')
    for bar, p in zip(bars, prob.cpu().numpy()):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{p:.2%}', ha='center', fontsize=11)
    plt.tight_layout(); plt.show()

# Test on a fake image
predict_mri(sorted(FAKE_DIR.glob('*.png'))[42], vit_scratch, 'best_vit_scratch.pth')

## 📚 STEP 13 — ViT Architecture Summary & Next Steps

```
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 ViT ARCHITECTURE YOU BUILT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Input     → 224×224×3 MRI image
 Patches   → 196 patches of 16×16 px
 Embedding → Linear projection to D=256
 Tokens    → 197 (196 patches + 1 [CLS])
 Encoder   → 6 × TransformerBlock
               ├─ LayerNorm
               ├─ MultiHeadSelfAttention (8 heads)
               ├─ LayerNorm
               └─ MLP (GELU, 4× expansion)
 Head      → [CLS] → Linear(256→128) → Linear(128→2)
 Output    → Real / Fake probability

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 RESEARCH UPGRADES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 1. Use real data
    BraTS 2023  → https://www.synapse.org/brats2023
    SynthRAD    → https://synthrad2023.grand-challenge.org

 2. Bigger pretrained ViT
    timm.create_model('vit_base_patch16_224',  pretrained=True)
    timm.create_model('vit_large_patch16_224', pretrained=True)

 3. Medical ViT variants
    MedViT, TransMed, Swin-UNet (hierarchical)

 4. 3D ViT for volumetric MRI
    Patch 3D slices using 3D convolutions

 5. Frequency analysis
    FFT the MRI → GAN artifacts appear in high-freq bands

 6. Advanced training
    Mixup, CutMix, MAE pretraining on unlabeled MRIs
```